# Familiarity vs. Answerability: Colab Execution

This notebook orchestrates the preregistered pipeline. It contains no estimators, scientific scoring logic, or claim decisions. Protected endpoints remain closed until their dedicated CLI transactions are available.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ.get("HF_TOKEN"), "Add HF_TOKEN to Colab Secrets before model access."
gpu_query = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True,
).strip().splitlines()[0]
gpu_name, gpu_memory_mib = [value.strip() for value in gpu_query.rsplit(",", 1)]
disk = shutil.disk_usage("/content")
preflight = {
    "gpu": gpu_name,
    "gpu_gib": round(int(gpu_memory_mib) / 1024, 2),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2),
    "disk_free_gib": round(disk.free / 2**30, 2),
}
assert preflight["gpu_gib"] >= 14 and preflight["disk_free_gib"] >= 40, preflight
preflight


In [ ]:
drive.mount("/content/drive")
REPO = Path("/content/mechanistic-interpretability")
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/fa-study-checkpoints")
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
launch_manifest_path = Path(os.environ.get("FA_LAUNCH_MANIFEST", str(DRIVE_CHECKPOINT_ROOT / "fa-study-launch.json")))
launch = json.loads(launch_manifest_path.read_text(encoding="utf-8")) if launch_manifest_path.is_file() else {}
if launch:
    assert set(launch) == {"schema_version", "git_commit", "bundle_file", "bundle_sha256"}
    assert launch["schema_version"] == 1
    assert Path(launch["bundle_file"]).name == launch["bundle_file"]
expected_commit = os.environ.get("FA_GIT_COMMIT") or launch.get("git_commit")
assert expected_commit, "Set FA_GIT_COMMIT to the frozen 40-character commit."
bundle_default = DRIVE_CHECKPOINT_ROOT / launch.get("bundle_file", "fa-study.bundle")
bundle_path = Path(os.environ.get("FA_GIT_BUNDLE", str(bundle_default)))
expected_bundle_sha256 = os.environ.get("FA_GIT_BUNDLE_SHA256") or launch.get("bundle_sha256")
assert bundle_path.is_file(), f"Missing pinned Git bundle: {bundle_path}"
assert expected_bundle_sha256, "Set FA_GIT_BUNDLE_SHA256 before cloning the bundle."
observed_bundle_sha256 = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
assert observed_bundle_sha256 == expected_bundle_sha256, (observed_bundle_sha256, expected_bundle_sha256)
if not REPO.is_dir():
    subprocess.run(["git", "clone", str(bundle_path), str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "--detach", expected_commit], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == expected_commit, (actual_commit, expected_commit)
subprocess.run(["git", "diff", "--quiet"], check=True)
subprocess.run(["git", "diff", "--cached", "--quiet"], check=True)
untracked = subprocess.check_output(["git", "ls-files", "--others", "--exclude-standard"], text=True).splitlines()
unexpected = [path for path in untracked if not path.startswith("runs/familiarity_answerability/")]
assert not unexpected, f"Refusing unexpected untracked files: {unexpected}"
subprocess.run(["git", "bundle", "verify", str(bundle_path)], check=True)


## Pinned environment

Install only the core lock. The optional circuit profile is not part of the confirmatory run.


In [ ]:
VENV = Path("/content/fa-venv")
VENV_PYTHON = VENV / "bin/python"
if VENV.exists():
    shutil.rmtree(VENV)
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "-r", "requirements/fa-core.lock"], check=True)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
lock_bytes = Path("requirements/fa-core.lock").read_bytes()
pip_check = subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "check"],
    check=True,
    capture_output=True,
    text=True,
)
assert pip_check.stdout.strip() == "No broken requirements found.", pip_check.stdout
environment_probe = subprocess.run(
    [str(VENV_PYTHON), "-m", "trajectory_extractor.fa_colab_entrypoint", "fa-colab-preflight", "--root", str(REPO), "--lock", "requirements/fa-core.lock"],
    check=True,
    capture_output=True,
    text=True,
)
print(environment_probe.stderr)
print(environment_probe.stdout)
runtime_observation = json.loads(environment_probe.stdout.strip().splitlines()[-1])
assert runtime_observation["status"] == "ready", runtime_observation
print("fa-core.lock sha256:", hashlib.sha256(lock_bytes).hexdigest())


In [ ]:
CONFIG = "configs/familiarity_answerability_gemma2_2b.json"
ROOT = str(REPO)

def run_cli(*arguments: str, allow_error: bool = False) -> dict:
    command = [str(VENV_PYTHON), "-m", "trajectory_extractor.fa_colab_entrypoint", *arguments]
    print(" ".join(command))
    completed = subprocess.run(command, check=False, capture_output=True, text=True)
    if completed.stderr.strip():
        print(completed.stderr)
    print(completed.stdout)
    payload = json.loads(completed.stdout.strip().splitlines()[-1])
    payload["_returncode"] = completed.returncode
    if completed.returncode != 0 and not allow_error:
        raise RuntimeError(f"CLI transaction failed: {payload}")
    return payload


## Frozen Source-v5 factual screening

This phase qualifies the preregistered source pool only. It generates 1,152 factual-screening completions and assembles exactly 244 screened real/synthetic pairs. It does not materialize F1/F2A prompts or open a protected endpoint. Each split is verified and checkpointed independently.


In [ ]:
screening_arguments = [
    "fa-run-colab-screening",
    "--config", CONFIG,
    "--root", ROOT,
    "--checkpoint-root", str(DRIVE_CHECKPOINT_ROOT),
    "--scratch-root", "/content",
    "--git-commit", actual_commit,
    "--bundle-path", str(bundle_path),
    "--bundle-sha256", expected_bundle_sha256,
]
if launch:
    screening_arguments.extend(["--launch-manifest", str(launch_manifest_path)])


In [ ]:
assembled = run_cli(*screening_arguments)
assert assembled["_returncode"] == 0, assembled
assert assembled["status"] == "assembled", assembled
assert assembled["source_candidate_count"] == 384, assembled
assert assembled["screening_completion_count"] == 1152, assembled
assert assembled["count"] == 244, assembled
assert assembled["protected_endpoints_accessed"] is False, assembled
assert assembled["stopped_before"] == "naturalness_and_f1_f2a", assembled
collection_manifest = Path(assembled["manifest"])
print("Source-v5 screening complete:", collection_manifest)


## Deliberate protocol stop

Do not continue into task construction or protected F1/F2A execution. First export the blinded naturalness packets and obtain two independent human ratings, with a third independent adjudicator available for disagreements.


In [ ]:
STOP_AFTER_SCREENING_ASSEMBLY = True
assert not STOP_AFTER_SCREENING_ASSEMBLY, (
    "Intentional stop: complete the preregistered two-rater naturalness gate before later cells."
)


## Resumable core sequence

Set the paths below only to verified artifacts from the same run. `fa-materialize-probe-rows` creates compact, provenance-bound evidence from generation, registered activations, exact teacher-forced scores, metadata, and outcomes. Protected rows remain unreadable to selection code.


In [ ]:
run_cli("fa-audit-manifest", "--config", CONFIG, "--root", ROOT, "--manifest", "<verified-manifest>")
run_cli("fa-materialize-probe-rows", "--config", CONFIG, "--root", ROOT, "--namespace", "mechanism_train", "--manifest", "<mechanism-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "mechanism-0000", "--resume")
run_cli("fa-materialize-probe-rows", "--config", CONFIG, "--root", ROOT, "--namespace", "locked_validation", "--manifest", "<validation-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "validation-0000", "--resume")
run_cli("fa-fit-probes", "--config", CONFIG, "--root", ROOT, "--train-rows-manifest", "<mechanism-probe-rows>", "--validation-rows-manifest", "<validation-probe-rows>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--shard-id", "selection-0000")
run_cli("fa-seal-behavior-test", "--config", CONFIG, "--root", ROOT, "--behavior-test-manifest", "<behavior-test-prompt-manifest>")
run_cli("fa-seal-selection", "--config", CONFIG, "--root", ROOT, "--selection-manifest", "<f2a-selection-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>")
run_cli("fa-evaluate-behavior-test", "--config", CONFIG, "--root", ROOT, "--manifest", "<behavior-test-prompt-manifest>", "--shard-id", "behavior-0000")
run_cli("fa-evaluate-probe-test", "--config", CONFIG, "--root", ROOT, "--selection-manifest", "<f2a-selection-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "probe-test-0000")
run_cli("fa-build-report", "--config", CONFIG, "--root", ROOT, "--behavior-test-manifest", "<behavior-test-prompt-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--selection-manifest", "<f2a-selection-manifest>", "--output", "reports/familiarity_answerability.md")


## Stop conditions

Stop on any pin, hash, audit, OOM, completion, or endpoint-state failure. Run transactions on local Colab storage. After each successful transaction, checkpoint only completed checksum-verified shards to `DRIVE_CHECKPOINT_ROOT`; restore only artifacts that pass their sidecar verification. A runtime interruption resumes from verified shard manifests rather than filenames. The notebook does not treat Google Drive writes as atomic.
